# Fase 5 — Votación de Transformers (BETO + DistilBETO + RoBERTuito, v6)

Este notebook **no entrena nada ni carga checkpoints**. Combina las probabilidades que ya guardó cada notebook v6, así que puede ejecutarse en cualquier máquina que tenga la carpeta `data/` con los resultados de los tres modelos (archivos pequeños, < 1 MB por modelo):

| Archivo (por modelo) | Para qué se usa |
|---|---|
| `diag_<modelo>_diagnostico/<var>_inner{1,2,3}_logits_refit.npz` | Logits **fuera de fold del train** (configuración final, 3 folds que cubren todo el train) → **validación** de la votación |
| `<modelo>_temperatura.json` | Temperatura ajustada **solo con el train** → probabilidades calibradas |
| `<modelo>_test_probabilidades.csv` | Probabilidades del **test oficial** por semilla → evaluación final (bloqueada) |

**Dos reglas de votación, fijadas antes de ver resultados (sin pesos y con umbral 0.5):**

- **Soft voting (principal):** promedio de las probabilidades calibradas de los tres modelos; irónico si el promedio ≥ 0.5.
- **Hard voting (secundaria):** cada modelo vota con su predicción; irónico si al menos 2 de 3 votan irónico (igual que la votación de ML).

Referencias: el sistema ganador de SemEval-2022 iSarcasmEval combinó votación *hard* y *soft* de tres Transformers (Abu Farha et al., 2022); Du et al. (2022) y Hu et al. (2022) usan ensambles de Transformers en SemEval-2022.

**Orden:** (1) validación con el train; (2) test oficial **solo si** `EJECUTAR_TEST_VOTACION = True` y `CONFIRMACION_TEST = "TEST_OFICIAL_CONGELADO"`. Si se ejecuta, es una evaluación adicional del test y debe declararse; se corre una sola vez.

## 1. Librerías

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import f1_score
import sklearn

print("numpy", np.__version__, "| pandas", pd.__version__, "| sklearn", sklearn.__version__)

## 2. Configuración

In [ ]:
DATA_DIR = Path("../data")
OUT_DIR = DATA_DIR / "resultados_votacion_transformers_v6"
OUT_DIR.mkdir(parents=True, exist_ok=True)

VARIANTES = ["mx", "es", "cu"]
MODELOS = {
    "BETO": (DATA_DIR / "resultados_beto_grouped_5x3_tpe20_v6", "beto"),
    "DistilBETO": (DATA_DIR / "resultados_distilbeto_grouped_5x3_tpe20_v6", "distilbeto"),
    "RoBERTuito": (DATA_DIR / "resultados_robertuito_grouped_5x3_tpe20_v6", "robertuito"),
}
MEJOR_INDIVIDUAL = "RoBERTuito"   # referencia de comparación (mejor modelo en la Nested CV)
UMBRAL = 0.5
N_BOOTSTRAP = 2000
SEMILLA_BOOTSTRAP = 42

# Test oficial: bloqueado por defecto.
EJECUTAR_TEST_VOTACION = False
CONFIRMACION_TEST = ""

for nombre, (carpeta, pref) in MODELOS.items():
    faltan = [p for p in [carpeta / f"{pref}_temperatura.json", carpeta / f"{pref}_test_probabilidades.csv"]
              + [carpeta / f"diag_{pref}_diagnostico" / f"{v}_inner{k}_logits_refit.npz" for v in VARIANTES for k in (1, 2, 3)]
              if not p.exists()]
    print(f"{nombre:<11} {'OK' if not faltan else 'FALTAN: ' + ', '.join(map(str, faltan))}")
print("Salida:", OUT_DIR.resolve())

## 3. Validación con el train (predicciones fuera de fold)

Para cada modelo se juntan los 3 folds del diagnóstico (cada tuit tiene una predicción de un modelo que **no** lo vio al entrenar), se aplica su temperatura y se combinan tuit por tuit. Los folds no son idénticos entre modelos, pero eso no afecta: cada probabilidad es fuera de fold.

Se reporta el F1-Macro de cada modelo y de las dos votaciones, la diferencia con RoBERTuito y un intervalo bootstrap pareado del 95 % (remuestreo de tuits). *Nota:* el diagnóstico usa la configuración final elegida con todo el train, así que los valores son algo optimistas **para todos por igual**; sirven para comparar, no como estimación final (esa es la Nested CV).

In [ ]:
def softmax_irony(logits, T):
    z = logits / T
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e[:, 1] / e.sum(axis=1)

def f1m(y, p):
    return f1_score(y, p, average="macro")

def cargar_oof(variante):
    probs, y = {}, None
    for nombre, (carpeta, pref) in MODELOS.items():
        T = json.load(open(carpeta / f"{pref}_temperatura.json"))[variante]["T"]
        idx, lg, lb = [], [], []
        for k in (1, 2, 3):
            z = np.load(carpeta / f"diag_{pref}_diagnostico" / f"{variante}_inner{k}_logits_refit.npz")
            idx.append(z["val_idx"]); lg.append(z["logits"]); lb.append(z["labels"])
        idx = np.concatenate(idx); orden = np.argsort(idx)
        assert len(np.unique(idx)) == len(idx), f"{nombre} {variante}: índices repetidos"
        lb = np.concatenate(lb)[orden]
        if y is None:
            y, idx_ref = lb, idx[orden]
        else:
            assert np.array_equal(idx[orden], idx_ref) and np.array_equal(lb, y), f"{nombre} {variante}: no coincide con los otros modelos"
        probs[nombre] = softmax_irony(np.concatenate(lg)[orden], T)
    return y, probs

def votar(probs):
    soft = (np.mean(list(probs.values()), axis=0) >= UMBRAL).astype(int)
    votos = sum((p >= UMBRAL).astype(int) for p in probs.values())
    hard = (votos >= 2).astype(int)
    return soft, hard

filas, rng = [], np.random.default_rng(SEMILLA_BOOTSTRAP)
acuerdos = []
for v in VARIANTES:
    y, probs = cargar_oof(v)
    preds = {m: (p >= UMBRAL).astype(int) for m, p in probs.items()}
    preds["Soft voting"], preds["Hard voting"] = votar(probs)
    ref = preds[MEJOR_INDIVIDUAL]
    boot = rng.integers(0, len(y), size=(N_BOOTSTRAP, len(y)))
    for m, p in preds.items():
        fila = {"variante": v, "modelo": m, "n": len(y), "f1_macro": f1m(y, p)}
        if m != MEJOR_INDIVIDUAL:
            d = np.array([f1m(y[b], p[b]) - f1m(y[b], ref[b]) for b in boot])
            fila.update(dif_vs_mejor=fila["f1_macro"] - f1m(y, ref),
                        ic95_inf=np.percentile(d, 2.5), ic95_sup=np.percentile(d, 97.5),
                        prop_boot_mejor=(d > 0).mean())
        filas.append(fila)
    nombres = list(MODELOS)
    for i in range(3):
        for j in range(i + 1, 3):
            acuerdos.append({"variante": v, "par": f"{nombres[i]}–{nombres[j]}",
                             "acuerdo": (preds[nombres[i]] == preds[nombres[j]]).mean()})

val = pd.DataFrame(filas)
val.to_csv(OUT_DIR / "votacion_validacion_train_oof.csv", index=False)
ac = pd.DataFrame(acuerdos)
ac.to_csv(OUT_DIR / "votacion_acuerdo_modelos.csv", index=False)

display(val.pivot(index="modelo", columns="variante", values="f1_macro")[VARIANTES]
        .reindex([*MODELOS, "Soft voting", "Hard voting"]).round(4))
display(val[val.modelo.str.contains("voting")][["variante", "modelo", "dif_vs_mejor", "ic95_inf", "ic95_sup", "prop_boot_mejor"]].round(4))
display(ac.pivot(index="par", columns="variante", values="acuerdo")[VARIANTES].round(3))

**Cómo leer la tabla:** `dif_vs_mejor` > 0 significa que la votación supera a RoBERTuito. Si el intervalo `ic95` incluye el 0, la diferencia no es concluyente. Un acuerdo alto entre modelos (> 0.8) indica que se equivocan en los mismos tuits, por lo que votar aporta poco.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
orden = [*MODELOS, "Soft voting", "Hard voting"]
colores = ["#1f77b4", "#ff7f0e", "#2ca02c", "#9467bd", "#e377c2"]
for a, v in zip(ax, VARIANTES):
    d = val[val.variante == v].set_index("modelo").loc[orden, "f1_macro"]
    a.bar(range(len(orden)), d.values, color=colores)
    a.axhline(d[MEJOR_INDIVIDUAL], color="#2ca02c", ls=":", lw=1)
    for k, x in enumerate(d.values):
        a.text(k, x + 0.003, f"{x:.3f}", ha="center", fontsize=8)
    a.set_xticks(range(len(orden)), orden, rotation=30, ha="right")
    a.set_title(v.upper()); a.grid(axis="y", alpha=.3)
lo = val.f1_macro.min()
ax[0].set_ylim(lo - 0.03, val.f1_macro.max() + 0.02)
ax[0].set_ylabel("F1-Macro (train, fuera de fold)")
fig.suptitle("Votación de Transformers — validación con el train", y=1.02)
fig.tight_layout()
fig.savefig(OUT_DIR / "votacion_validacion.png", dpi=200, bbox_inches="tight")
fig.savefig(OUT_DIR / "votacion_validacion.pdf", bbox_inches="tight")
plt.show()

## 4. Test oficial (bloqueado)

Para cada semilla *s* se combinan las predicciones de la semilla *s* de los tres modelos (5 votaciones por variante). Soft voting usa `prob_ironia_calibrada`; hard voting usa `y_pred` de cada modelo. Se reporta media ± d.e. entre semillas, F1 de la clase irónica y F1 sin los textos que coinciden con el train.

Para ejecutarlo: `EJECUTAR_TEST_VOTACION = True` y `CONFIRMACION_TEST = "TEST_OFICIAL_CONGELADO"` en la sección 2. **Una sola vez.**

In [ ]:
if not (EJECUTAR_TEST_VOTACION and CONFIRMACION_TEST == "TEST_OFICIAL_CONGELADO"):
    print("Test oficial BLOQUEADO. Se reportan solo los resultados de validación.")
else:
    tablas = []
    for nombre, (carpeta, pref) in MODELOS.items():
        d = pd.read_csv(carpeta / f"{pref}_test_probabilidades.csv")
        d["modelo"] = nombre
        tablas.append(d)
    test = pd.concat(tablas, ignore_index=True)
    filas = []
    for v in VARIANTES:
        for s in sorted(test.seed.unique()):
            sub = test[(test.variante == v) & (test.seed == s)]
            w = sub.pivot(index="ID", columns="modelo", values=["prob_ironia_calibrada", "y_pred", "y_true", "coincide_train"])
            assert len(w) == 600 and w["y_true"].nunique(axis=1).eq(1).all(), f"{v} semilla {s}: IDs o etiquetas no coinciden"
            y = w["y_true"].iloc[:, 0].astype(int).values
            limpio = ~w["coincide_train"].iloc[:, 0].astype(bool).values
            preds = {m: w["y_pred"][m].astype(int).values for m in MODELOS}
            preds["Soft voting"] = (w["prob_ironia_calibrada"][list(MODELOS)].mean(axis=1).values >= UMBRAL).astype(int)
            preds["Hard voting"] = (sum(preds[m] for m in MODELOS) >= 2).astype(int)
            for m, p in preds.items():
                filas.append({"variante": v, "seed": s, "modelo": m,
                              "f1_macro": f1m(y, p),
                              "f1_ironia": f1_score(y, p, pos_label=1),
                              "f1_macro_sin_coincidencias": f1m(y[limpio], p[limpio])})
    por_semilla = pd.DataFrame(filas)
    por_semilla.to_csv(OUT_DIR / "votacion_test_por_semilla.csv", index=False)
    resumen = (por_semilla.groupby(["variante", "modelo"])
               .agg(f1_macro_mean=("f1_macro", "mean"), f1_macro_std=("f1_macro", "std"),
                    f1_ironia_mean=("f1_ironia", "mean"),
                    f1_sin_coinc_mean=("f1_macro_sin_coincidencias", "mean"))
               .reset_index())
    resumen.to_csv(OUT_DIR / "votacion_test_resumen.csv", index=False)
    t = resumen.pivot(index="modelo", columns="variante", values="f1_macro_mean")[VARIANTES]
    t["general"] = t.mean(axis=1)
    display(t.reindex([*MODELOS, "Soft voting", "Hard voting"]).round(4))
    display(resumen.round(4))

## 5. Interpretación (completar tras ejecutar)

- Si la votación **no supera** a RoBERTuito en validación: los modelos están muy correlacionados (acuerdo > 0.8; DistilBETO es una destilación de BETO) y RoBERTuito es claramente superior, así que los otros dos corrigen menos errores de los que introducen. Se reporta como resultado y el modelo final sigue siendo RoBERTuito.
- Si la **supera** de forma consistente (IC 95 % por encima de 0 en al menos 2 de 3 variantes): se puede evaluar en el test, declarándolo como evaluación adicional.